# SIH26006 — Phase 8: Deep Learning (GRU & LSTM) Experiment

This self-contained notebook runs the **Phase 8 GRU and LSTM final experiments** on Google Colab GPU.

### How to Open & Run in Colab:
1. **Opening Notebook**: In Colab **File -> Upload notebook**, upload `phase8_gru_lstm_colab.ipynb` (the `.ipynb` file, NOT a `.zip` file).
2. **Loading Project Data** (Choose Method A or B):
   - **Method A (GitHub Clone - Recommended)**: Set `GITHUB_REPO_URL` in Cell 1 to your repo URL and run!
   - **Method B (Zip Upload)**: Drag & drop your project ZIP into Colab's left sidebar file manager (📁 icon).

### GPU Accelerator:
Ensure GPU is enabled (**Runtime -> Change runtime type -> T4 GPU**).

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1: GPU DETECTION & AUTOMATIC REPO CLONE TO ROOT
# ─────────────────────────────────────────────────────────────
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

print('TensorFlow Version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU AVAILABLE: {len(gpus)} GPU device(s) detected.')
    for gpu in gpus:
        print(f'   Device Name: {gpu.name}')
        try:
            details = tf.config.experimental.get_device_details(gpu)
            if 'device_name' in details:
                print(f'   GPU Model: {details["device_name"]}')
        except Exception:
            pass
else:
    print('⚠️ WARNING: No GPU detected! Execution will fall back to CPU.')

# Clone repo and copy files directly into root working directory (/content/)
if not os.path.exists('outputs/modeling_dataset.csv'):
    print('🌐 Auto-cloning repository from GitHub...')
    !git clone https://github.com/SSOHEB/FICOS-Platform.git repo_temp
    !cp -r repo_temp/* .
    !rm -rf repo_temp
    print('✅ Repository files copied to environment root.')

# Create output directories
os.makedirs('outputs/predictions/gru', exist_ok=True)
os.makedirs('outputs/predictions/lstm', exist_ok=True)
os.makedirs('outputs/plots/deep_learning', exist_ok=True)

if os.path.exists('outputs/modeling_dataset.csv'):
    print('✅ SUCCESS: outputs/modeling_dataset.csv is ready!')
else:
    print('❌ ERROR: outputs/modeling_dataset.csv missing.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2: DATA PROCESSING & 30-DAY SEQUENCE CONSTRUCTION
# ─────────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

# Set fixed random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Robust multi-path dataset locator
possible_paths = [
    'outputs/modeling_dataset.csv',
    'FICOS-Platform/outputs/modeling_dataset.csv',
    '/content/FICOS-Platform/outputs/modeling_dataset.csv',
    '/content/outputs/modeling_dataset.csv'
]

dataset_path = None
for p in possible_paths:
    if os.path.exists(p):
        dataset_path = p
        break

# Automatic inline clone if dataset isn't found in any path
if dataset_path is None:
    print('🌐 Auto-cloning FICOS-Platform repository...')
    !git clone https://github.com/SSOHEB/FICOS-Platform.git FICOS-Platform
    for p in possible_paths:
        if os.path.exists(p):
            dataset_path = p
            break

print(f'✅ Loading dataset from: {dataset_path}')
df = pd.read_csv(dataset_path)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Chronological 70 / 15 / 15 Split
n = len(df)
n_train = int(n * 0.70)
n_val = int(n * 0.15)
n_test = n - n_train - n_val

print(f'Total Rows: {n} | Train: {n_train} | Val: {n_val} | Test: {n_test}')
print(f'Train Dates: {df["date"].iloc[0].strftime("%Y-%m-%d")} -> {df["date"].iloc[n_train-1].strftime("%Y-%m-%d")}')
print(f'Val Dates:   {df["date"].iloc[n_train].strftime("%Y-%m-%d")} -> {df["date"].iloc[n_train+n_val-1].strftime("%Y-%m-%d")}')
print(f'Test Dates:  {df["date"].iloc[n_train+n_val].strftime("%Y-%m-%d")} -> {df["date"].iloc[-1].strftime("%Y-%m-%d")}')

feature_cols = [c for c in df.columns if not c.startswith('target_') and c not in ['date']]
TARGETS = ['kdci', 'cape', 'panamax', 'supramax', 'handy']
HORIZONS = [1, 7, 14, 30]

def create_sequences(data_df, feature_cols, target_col, prev_col, lookback=30):
    '''Creates 30-day lookback sequence windows (X: N x 30 x num_features, y: N)'''
    X_list, y_list, y_prev_list, date_list = [], [], [], []
    feat_vals = data_df[feature_cols].values
    tgt_vals = data_df[target_col].values
    prev_vals = data_df[prev_col].values
    dates = data_df['date'].values
    
    for i in range(lookback, len(data_df)):
        if np.isnan(tgt_vals[i]) or np.isnan(prev_vals[i]):
            continue
        X_seq = feat_vals[i-lookback:i, :]
        if np.isnan(X_seq).any():
            continue
        X_list.append(X_seq)
        y_list.append(tgt_vals[i])
        y_prev_list.append(prev_vals[i])
        date_list.append(dates[i])
        
    return np.array(X_list), np.array(y_list), np.array(y_prev_list), np.array(date_list)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3: KERAS GRU & LSTM ARCHITECTURE BUILDERS
# ─────────────────────────────────────────────────────────────
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

def build_gru_model(input_shape):
    model = Sequential([
        GRU(32, input_shape=input_shape, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(32, input_shape=input_shape, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4: MODEL TRAINING LOOP (20 GRU + 20 LSTM MODELS)
# ─────────────────────────────────────────────────────────────
def calc_metrics(y_true, y_pred, y_prev):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    smape = np.mean(200 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    ss_res = np.sum((y_true - y_pred)**2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
    actual_dir = np.sign(y_true - y_prev)
    pred_dir = np.sign(y_pred - y_prev)
    dir_acc = np.mean(actual_dir == pred_dir)
    return round(mae, 2), round(rmse, 2), round(smape, 2), round(r2, 4), round(dir_acc, 4)

deep_results = []

print('Starting Phase 8 Training Loop across 5 Targets x 4 Horizons...')

for tgt in TARGETS:
    for h in HORIZONS:
        target_col = f'target_{tgt}_{h}d'
        prev_col = tgt
        
        # Split data
        train_df = df.iloc[:n_train].copy()
        val_df = df.iloc[n_train:n_train+n_val].copy()
        test_df = df.iloc[n_train+n_val:].copy()
        
        # Impute missing feature values using train median ONLY
        train_medians = train_df[feature_cols].median()
        train_df[feature_cols] = train_df[feature_cols].fillna(train_medians)
        val_df[feature_cols] = val_df[feature_cols].fillna(train_medians)
        test_df[feature_cols] = test_df[feature_cols].fillna(train_medians)
        
        # Scale features using scaler fit ONLY on training data
        scaler = StandardScaler()
        train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
        val_df[feature_cols] = scaler.transform(val_df[feature_cols])
        test_df[feature_cols] = scaler.transform(test_df[feature_cols])
        
        # Create 30-day lookback sequences
        X_tr, y_tr, _, _ = create_sequences(train_df, feature_cols, target_col, prev_col, lookback=30)
        X_v, y_v, _, _ = create_sequences(val_df, feature_cols, target_col, prev_col, lookback=30)
        X_te, y_te, y_prev_te, dates_te = create_sequences(test_df, feature_cols, target_col, prev_col, lookback=30)
        
        if len(X_te) == 0:
            continue

        input_shape = (X_tr.shape[1], X_tr.shape[2])
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        
        # 1. Train GRU
        gru = build_gru_model(input_shape)
        gru.fit(X_tr, y_tr, validation_data=(X_v, y_v), epochs=50, batch_size=32, callbacks=[early_stop], verbose=0)
        pred_g = gru.predict(X_te, verbose=0).flatten()
        mae_g, rmse_g, smape_g, r2_g, dacc_g = calc_metrics(y_te, pred_g, y_prev_te)
        
        pd.DataFrame({'date': dates_te, 'y_true': y_te, 'y_pred': pred_g, 'y_prev': y_prev_te}).to_csv(
            f'outputs/predictions/gru/{tgt}_{h}d.csv', index=False)
        
        deep_results.append({
            'model': 'GRU', 'freight_class': tgt, 'horizon': f'{h}d',
            'MAE': mae_g, 'RMSE': rmse_g, 'sMAPE': smape_g, 'R2': r2_g,
            'directional_accuracy': dacc_g, 'n_test': len(y_te)
        })
        
        # 2. Train LSTM
        lstm = build_lstm_model(input_shape)
        lstm.fit(X_tr, y_tr, validation_data=(X_v, y_v), epochs=50, batch_size=32, callbacks=[early_stop], verbose=0)
        pred_l = lstm.predict(X_te, verbose=0).flatten()
        mae_l, rmse_l, smape_l, r2_l, dacc_l = calc_metrics(y_te, pred_l, y_prev_te)
        
        pd.DataFrame({'date': dates_te, 'y_true': y_te, 'y_pred': pred_l, 'y_prev': y_prev_te}).to_csv(
            f'outputs/predictions/lstm/{tgt}_{h}d.csv', index=False)
        
        deep_results.append({
            'model': 'LSTM', 'freight_class': tgt, 'horizon': f'{h}d',
            'MAE': mae_l, 'RMSE': rmse_l, 'sMAPE': smape_l, 'R2': r2_l,
            'directional_accuracy': dacc_l, 'n_test': len(y_te)
        })
        
        print(f'  {tgt.upper()} {h:2d}d | GRU MAE: {mae_g:7.2f} (R2: {r2_g:6.4f}, Dir: {dacc_g:.1%}) | LSTM MAE: {mae_l:7.2f} (R2: {r2_l:6.4f}, Dir: {dacc_l:.1%})')

df_deep = pd.DataFrame(deep_results)
df_deep.to_csv('outputs/deep_model_comparison.csv', index=False)
print('\n✅ Saved Phase 8 results to outputs/deep_model_comparison.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5: BENCHMARK COMPARISON MATRIX (PERSISTENCE vs RIDGE vs XGBOOST vs GRU vs LSTM)
# ─────────────────────────────────────────────────────────────
comp_paths = [
    'outputs/model_comparison.csv',
    'FICOS-Platform/outputs/model_comparison.csv',
    '/content/FICOS-Platform/outputs/model_comparison.csv'
]
comp_file = next((p for p in comp_paths if os.path.exists(p)), None)

if comp_file:
    base_results = pd.read_csv(comp_file)
    full_comp = pd.concat([base_results, df_deep], ignore_index=True)
    full_comp.to_csv('outputs/full_5model_comparison.csv', index=False)
    print(f'✅ Loaded baseline metrics from {comp_file}')
    print('✅ Saved 5-Model Comparison Matrix to outputs/full_5model_comparison.csv')
    
    print('\n' + '='*75)
    print('BENCHMARK EVALUATION: DOES DEEP LEARNING (GRU/LSTM) BEAT RIDGE OR PERSISTENCE?')
    print('='*75)
    
    for tgt in TARGETS:
        print(f'\n  --- {tgt.upper()} ---')
        for h in HORIZONS:
            h_str = f'{h}d'
            sub = full_comp[(full_comp['freight_class'] == tgt) & (full_comp['horizon'] == h_str)]
            if sub.empty:
                continue
            best_row = sub.loc[sub['MAE'].idxmin()]
            pers_rows = sub[sub['model'] == 'Persistence']
            if not pers_rows.empty:
                pers_mae = pers_rows.iloc[0]['MAE']
                imp_pct = ((pers_mae - best_row['MAE']) / pers_mae) * 100
                print(f'    {h_str:3s}: Winner={best_row["model"]:11s} | MAE={best_row["MAE"]:7.2f} | R2={best_row["R2"]:6.4f} | DirAcc={best_row["directional_accuracy"]:.1%} | (vs Persistence: {imp_pct:+.1f}%)')
            else:
                print(f'    {h_str:3s}: Winner={best_row["model"]:11s} | MAE={best_row["MAE"]:7.2f} | R2={best_row["R2"]:6.4f} | DirAcc={best_row["directional_accuracy"]:.1%}')
else:
    print('outputs/model_comparison.csv not found. Displaying GRU vs LSTM only:')
    print(df_deep.to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6: COMPARISON PLOT VISUALIZATIONS
# ─────────────────────────────────────────────────────────────
if 'full_comp' in locals():
    fig, axes = plt.subplots(len(TARGETS), 1, figsize=(12, 18), sharex=True)
    models_order = ['Persistence', 'Ridge', 'XGBoost', 'GRU', 'LSTM']
    
    for i, tgt in enumerate(TARGETS):
        ax = axes[i]
        sub = full_comp[full_comp['freight_class'] == tgt]
        pivot = sub.pivot(index='horizon', columns='model', values='MAE').reindex(['1d', '7d', '14d', '30d'])
        pivot = pivot[[m for m in models_order if m in pivot.columns]]
        
        pivot.plot(kind='bar', ax=ax, width=0.8)
        ax.set_title(f'MAE Comparison (All 5 Models) — {tgt.upper()}', fontsize=12, fontweight='bold')
        ax.set_ylabel('MAE ($/day)')
        ax.grid(True, alpha=0.3)
        ax.legend(title='Model', loc='upper left')
    
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig('outputs/plots/deep_learning/deep_model_comparison_mae.png', dpi=300)
    plt.show()
    print('Saved comparison plot: outputs/plots/deep_learning/deep_model_comparison_mae.png')